# Phase 4.1：HTTP、JSON 与 API 合同

## 目标

从用户视角理解服务：请求是什么，响应是什么，状态码表达什么，为什么 `/search` 必须返回 citations。先使用 FastAPI `TestClient` 在内存中验证，不依赖手工启动服务器。

**本课交付：** 一组 API 合同断言，为下一课的产品编排提供边界。

## Evidence Quest 任务卡：Phase 4.1：API 合同审讯室

**你的身份：** 产品接口审查员  
**案件背景：** 搜索能力要交给别人使用，接口就像一份对外承诺：输入什么、返回什么、错误如何表达，都不能靠猜。

### 本关专业 Goal

写清 HTTP、JSON、状态码和 citation contract，并用测试保护它。

### 你要交付的作品

**可验证的 API 合同清单**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：接口审查员  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. Python 函数和 HTTP 接口的区别

Python 函数可以直接传对象；HTTP 客户端只能传文本/JSON，并通过状态码告诉调用方结果。API 合同把内部对象转换成稳定的外部协议，前端、测试和其他服务都依赖这个协议。

本项目的核心接口：

```text
GET  /health  -> 服务和索引状态
POST /search  -> Query 和可追溯结果
POST /chat    -> 答案模式和 citations
```

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase4.1'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase4.1
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


## 2. 先手写一个最小 API 合同

在使用 FastAPI 之前，先用普通 Python 函数模拟一个服务边界。这个函数接收一个 JSON 风格字典，检查 `query`，再返回一个包含 `status_code/body` 的响应。它不是最终服务器，而是帮助你理解：API 的核心是稳定的输入校验和输出合同。

In [3]:
# 定义一个最小搜索函数，模拟 HTTP 服务的输入和输出。
def manual_search_api(request_body):
    # 从请求字典中读取 query；缺失时使用空字符串。
    query = str(request_body.get("query", ""))

    # 如果 query 为空，返回客户端输入错误的响应。
    if not query:
        return {"status_code": 422, "body": {"code": "QUERY_REQUIRED", "message": "query 不能为空"}}

    # 对合法输入返回一个固定结构，模拟后续检索结果。
    return {"status_code": 200, "body": {"query": query, "results": [], "citations": []}}

# 发送合法请求，观察服务合同的成功形状。
manual_ok = manual_search_api({"query": "overlap"})

# 打印成功响应的状态码和正文。
print(manual_ok)

# 发送空请求，观察输入校验如何转成 422。
manual_bad = manual_search_api({"query": ""})

# 打印错误响应，理解调用方可以根据 code 做什么。
print(manual_bad)

# 验证两类输入得到不同状态码。
assert manual_ok["status_code"] == 200
assert manual_bad["status_code"] == 422

{'status_code': 200, 'body': {'query': 'overlap', 'results': [], 'citations': []}}
{'status_code': 422, 'body': {'code': 'QUERY_REQUIRED', 'message': 'query 不能为空'}}


### 为什么还要使用 FastAPI？

手写函数展示了合同的本质，但没有 HTTP 路由、JSON 序列化、自动文档、类型校验和测试工具。FastAPI 把这些通用工程能力提供出来；我们要学习的是“知道它替我们做了什么”，而不是把框架当黑盒。

In [4]:
# 从 FastAPI 导入测试客户端，它会在内存中发送 HTTP 请求。
from fastapi.testclient import TestClient

# 从项目应用工厂导入 create_app，避免直接依赖全局状态。
from phase4_mini_rag_system.app import create_app

# 指定应用默认读取的教学输入目录。
input_directory = ROOT / "phase1_doc_parser" / "examples" / "input"

# 创建一个已经完成初始 ingest 的 FastAPI 应用。
app = create_app(input_directory)

# 创建测试客户端，后面的 get/post 就像真实 HTTP 调用。
client = TestClient(app)

# 确认客户端和应用对象创建成功。
print("API TestClient ready")

API TestClient ready


## 3. `/health`：先看服务是否准备好

健康接口通常返回 200 和索引信息。`chunks` 和 `index_version` 让调用方知道服务不是“能响应但没有数据”。

In [5]:
# 向 health 路由发送 GET 请求。
health_response = client.get("/health")

# 读取 HTTP 状态码。
print("status:", health_response.status_code)

# 把 JSON 响应转换为 Python 字典。
health_payload = health_response.json()

# 打印响应内容，观察外部 API 合同。
print(health_payload)

# 健康检查成功时必须返回 200。
assert health_response.status_code == 200

# 健康响应必须报告 Chunk 数和索引版本。
assert health_payload["chunks"] > 0
assert health_payload["index_version"]

status: 200
{'status': 'ok', 'chunks': 2, 'index_version': 'chunks-2-size-512-overlap-128'}


## 4. `/search`：请求和引用响应

请求体是 JSON：`query` 是用户问题，`top_k` 控制返回数量。响应中的每条结果必须包含 `chunk_id/text/source/page/score`，否则用户无法回到原文，也无法评估排名。

In [6]:
# 创建一个合法的搜索请求 JSON。
search_request = {"query": "Chunk overlap", "top_k": 3}

# 向 search 路由发送 POST 请求并携带 JSON。
search_response = client.post("/search", json=search_request)

# 把响应转换为 Python 字典。
search_payload = search_response.json()

# 输出状态和结果数量，理解请求到响应的转换。
print("status:", search_response.status_code)
print("result count:", len(search_payload["results"]))

# 合法请求必须返回 200。
assert search_response.status_code == 200

# 结果列表不能为空，否则 Query 没有得到证据。
assert search_payload["results"]

# 定义 citation 的最小字段集合。
citation_fields = {"chunk_id", "text", "source", "page", "score"}

# 检查第一条结果是否满足引用合同。
assert citation_fields <= search_payload["results"][0].keys()

status: 200
result count: 1


## 5. 状态码：输入错误不是服务器崩溃

Pydantic 会校验 `query` 的最小长度。空 Query 属于客户端输入错误，应返回 422；调用方可以据此提示用户，而不是显示“服务器坏了”。

In [7]:
# 创建一个违反 query 最小长度约束的请求。
invalid_request = {"query": ""}

# 发送非法请求，观察 FastAPI 的校验响应。
invalid_response = client.post("/search", json=invalid_request)

# 输出状态码和错误详情。
print("status:", invalid_response.status_code)
print("detail:", invalid_response.json().get("detail"))

# 空 Query 应该被识别为请求校验错误。
assert invalid_response.status_code == 422

status: 422
detail: [{'type': 'string_too_short', 'loc': ['body', 'query'], 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}}]


## 本课验收

- [ ] 能区分 Python 函数调用和 HTTP 请求。
- [ ] 能解释 200 与 422 的区别。
- [ ] 能说出 citations 为什么是产品合同而不是调试输出。
- [ ] 能手写一个最小请求校验和响应合同。
- [ ] `/health`、`/search` 和空 Query 行为已通过断言。

## Boss Challenge：为一个空 Query 或不存在的 source 写出预期状态码和错误 JSON。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [8]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [9]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase4_api_contract_record.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: 还没有生成，请回到交付单元格
